# make up publication year and filter nan

In [6]:
import os
import pickle
import numpy as np
import pandas as pd
import h5py
import mygene

In [ ]:

all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/disgenet_all_annotation.csv',sep=';')
all_df.head(3)
print(len(all_df))
all_df = all_df[~(all_df['uniport'] == '[]')]
print(len(all_df))

8737
8305


In [4]:
non_pub = all_df[all_df['first_pub_year'].isna()]

In [5]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
def get_earliest_pubmed_pmid_and_year(gene, disease):
    # Construct the search query
    query = f"{gene} AND {disease}"
    
    # PubMed URL to get the total number of results
    search_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={query}&retmode=json&sort=pub+date"
    
    try:
        # Fetch search results to get the total count
        response = requests.get(search_url)
        response.raise_for_status()
        total_results = int(response.json().get("esearchresult", {}).get("count", 0))

        if total_results == 0:
            print("No results found for the given query.")
            return None, None

        # Fetch only the last result (earliest publication)
        earliest_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={query}&retstart={total_results - 1}&retmax=1&retmode=json&sort=pub+date"
        earliest_response = requests.get(earliest_url)
        earliest_response.raise_for_status()
        pmid = earliest_response.json().get("esearchresult", {}).get("idlist", [])[0]

        # Fetch the publication year for the earliest publication
        summary_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pubmed&id={pmid}&retmode=json"
        summary_response = requests.get(summary_url)
        summary_response.raise_for_status()
        pub_year = summary_response.json()["result"][pmid]["pubdate"].split()[0]  # Extract the year

        return pmid, pub_year

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return None, None

# Define a function that will work with a row
def fetch_pubmed_data(row):
    pmid, pub_year = get_earliest_pubmed_pmid_and_year(row['gene_id'], row['disease_name'])
    return pd.Series(pub_year)

# Apply the function to each row
non_pub['first_pub_year'] = non_pub.apply(fetch_pubmed_data, axis=1)


No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results found for the given query.
No results f

/tmp/ipykernel_2892162/2018367235.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_pub['first_pub_year'] = non_pub.apply(fetch_pubmed_data, axis=1)


In [9]:
# filter out missing lines
len(non_pub),len(non_pub[non_pub['first_pub_year'].isna()]),len(non_pub[~(non_pub['first_pub_year'].isna())])

(3120, 413, 2707)

In [8]:
all_df_ori = all_df[~(all_df['first_pub_year'].isna())]
len(all_df_ori)

5185

In [10]:
dga_combined = pd.concat([all_df_ori, non_pub[~(non_pub['first_pub_year'].isna())]], ignore_index=True)

In [12]:
dga_combined.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/dga_all_pub.csv',index=False)

# generta ppi emb from two versions

In [ ]:

local_stringdb = os.path.join('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/2023')

ppidf = pd.read_csv(os.path.join(local_stringdb,'9606.protein.info.v12.0.txt'), sep='\t', header=0, usecols=['#string_protein_id', 'preferred_name'])
ppidf['preferred_name'] = ppidf['preferred_name'].str.upper()
stringId2name = ppidf.set_index('#string_protein_id')['preferred_name'].to_dict()
name2stringId = ppidf.set_index('preferred_name')['#string_protein_id'].to_dict()
ppidf = pd.read_csv(os.path.join(local_stringdb,'9606.protein.aliases.v12.0.txt'), sep='\t', header=0, usecols=['#string_protein_id', 'alias']).drop_duplicates(['alias'], keep='first')
ppidf['alias'] = ppidf['alias'].str.upper()
aliases2stringId = ppidf.set_index('alias')['#string_protein_id'].to_dict()

year = '2016'
full = True
local_stringdb = os.path.join('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb',year)
if full:
    # ppi_connection = pd.read_csv(os.path.join(local_stringdb,'9606.protein.links.v12.0.txt'), sep=' ', header=0).convert_dtypes().replace(0, float('nan'))
    ppi_connection = pd.read_csv(os.path.join(local_stringdb,'9606.protein.links.v10.txt'), sep=' ', header=0).convert_dtypes().replace(0, float('nan'))
    # ppi_connection = pd.read_csv(os.path.join(local_stringdb,'9606.protein.links.v10.5.txt'), sep=' ', header=0).convert_dtypes().replace(0, float('nan'))

    # ppi_connection = pd.read_csv(os.path.join(local_stringdb,'9606.protein.links.v9.1.txt'), sep=' ', header=0).convert_dtypes().replace(0, float('nan'))

else:
    ppi_connection = pd.read_csv(os.path.join(local_stringdb,'9606.protein.physical.links.full.v12.0.txt'), sep=' ', header=0).convert_dtypes().replace(0, float('nan'))

ppi_connection[['protein1', 'protein2']] = np.sort(ppi_connection[['protein1', 'protein2']], axis=1)
ppi_connection = ppi_connection.drop_duplicates()
ppi_connection

,protein1,protein2,combined_score
0,9606.ENSP00000000233,9606.ENSP00000003084,150
1,9606.ENSP00000000233,9606.ENSP00000003100,215
2,9606.ENSP00000000233,9606.ENSP00000005257,223
3,9606.ENSP00000000233,9606.ENSP00000005340,193
4,9606.ENSP00000000233,9606.ENSP00000006101,415
...,...,...,...
8537757,9606.ENSP00000472847,9606.ENSP00000473036,188
8537758,9606.ENSP00000472847,9606.ENSP00000473172,281
8538703,9606.ENSP00000472867,9606.ENSP00000472929,150
8540618,9606.ENSP00000472929,9606.ENSP00000473233,158


In [12]:
proteins = set(set(ppi_connection['protein1'].unique()) | set(ppi_connection['protein2'].unique()))
len(proteins)

19247

In [13]:
nodes_to_keep = set(proteins) & set(stringId2name.keys())
len(nodes_to_keep)

14298

In [14]:
ppi_connection = ppi_connection[ppi_connection['protein1'].isin(nodes_to_keep)]
ppi_connection = ppi_connection[ppi_connection['protein2'].isin(nodes_to_keep)]
len(ppi_connection)

2389568

In [15]:
ppi_connection[['protein1', 'protein2']].to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_clean_2017.txt', sep='\t', index=False, header=False)

In [16]:
### PPI features from node2vec
from pecanpy import pecanpy as node2vec

def Runnode2vec(filepath):
    n2v = node2vec.SparseOTF(p=1, q=1, workers=4, verbose=True)

    edge_list = n2v.read_edg(filepath, weighted=False, directed=False)
    emd = n2v.embed(dim=128, num_walks=10, walk_length=80, window_size=10, epochs=10)

    n2v_emd = pd.DataFrame(emd, n2v.nodes)

    n2v_emd.columns = ['network_' + str(col) for col in n2v_emd.columns]

    n2v_emd = n2v_emd.reset_index().rename(columns={"index":"string_id"})

    return n2v_emd

ppi_features = Runnode2vec('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_clean_2017.txt')

new_columns = ['string_id'] + [f'feature_{i}' for i, col in enumerate(ppi_features.columns) if col != 'string_id']
# Reorder the DataFrame so that 'string_id' is the first column
df_combined = ppi_features[['string_id'] + [col for col in ppi_features.columns if col != 'string_id']]
df_combined.columns = new_columns
df_combined.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_clean_2017_emb.csv', index=False)

  0%|          | 0/142780 [00:00<?, ?it/s]

# id maps

## ppi id mapping

In [4]:
def get_map_df(ensembl_ids,input_type):
    mg = mygene.MyGeneInfo()
    # Query mygene for UniProt and Entrez gene ID mappings
    results = mg.querymany(
        ensembl_ids,
        scopes=input_type,
        fields='uniprot,entrezgene',
        species='human'
    )

    results_df = pd.DataFrame(results)
    results_df = results_df[~results_df['entrezgene'].isna()]
    results_df['uniprot_ids'] = results_df['uniprot'].apply(
        lambda x: list(x.values())[0] if isinstance(x, dict) and 'Swiss-Prot' in x else None)
    results_df = results_df[~results_df['uniprot_ids'].isna()]
    results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]
    return results_df


# ppi_emb = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2019_emb.csv')
# ppi_ids_map = get_map_df(ppi_emb['string_id'].str.split('.').str[1],'ensembl.protein')

### check one esnp id to multiple uniport id cases

In [52]:
mg = mygene.MyGeneInfo()
# Query mygene for UniProt and Entrez gene ID mappings
results = mg.querymany(
    ppi_emb['string_id'].str.split('.').str[1],
    scopes='ensembl.protein',
    fields='uniprot,entrezgene',
    species='human'
)

results_df = pd.DataFrame(results)
results_df = results_df[~results_df['entrezgene'].isna()]


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found dup hits:	[('ENSP00000473163', 3)]
1939 input query terms found no hit:	['ENSP00000415070', 'ENSP00000267197', 'ENSP00000415452', 'ENSP00000006101', 'ENSP00000262477', 'ENS


In [59]:
results_df['uniprot_ids'] = results_df['uniprot'].apply(
    lambda x: x['Swiss-Prot'] if isinstance(x, dict) and 'Swiss-Prot' in x else None)


In [63]:
results_df[results_df['uniprot_ids'].apply(lambda x: len(x) > 1 if isinstance(x, list) else False)]

,query,_id,_score,entrezgene,uniprot,notfound,uniprot_ids
474,ENSP00000360141,2778,28.673742,2778,"{'Swiss-Prot': ['Q5JWF2', 'P84996', 'P63092', ...",NaN,"[Q5JWF2, P84996, P63092, O95467]"
941,ENSP00000272298,805,28.674032,805,"{'Swiss-Prot': ['P0DP24', 'P0DP23', 'P0DP25'],...",NaN,"[P0DP24, P0DP23, P0DP25]"
1722,ENSP00000251507,9910,28.674032,9910,"{'Swiss-Prot': ['Q5R372', 'B7ZAP0'], 'TrEMBL':...",NaN,"[Q5R372, B7ZAP0]"
1955,ENSP00000364802,3303,28.674032,3303,"{'Swiss-Prot': ['P0DMV9', 'P0DMV8'], 'TrEMBL':...",NaN,"[P0DMV9, P0DMV8]"
2221,ENSP00000291295,808,28.673742,808,"{'Swiss-Prot': ['P0DP24', 'P0DP23', 'P0DP25'],...",NaN,"[P0DP24, P0DP23, P0DP25]"
...,...,...,...,...,...,...,...
18622,ENSP00000377583,85294,29.599241,85294,"{'Swiss-Prot': ['Q9BYR9', 'P0C7H8']}",NaN,"[Q9BYR9, P0C7H8]"
18704,ENSP00000413896,100133093,29.599539,100133093,"{'Swiss-Prot': ['B3EWG5', 'B3EWG3', 'B3EWG6']}",NaN,"[B3EWG5, B3EWG3, B3EWG6]"
19517,ENSP00000392407,729246,29.599539,729246,"{'Swiss-Prot': ['P86481', 'P86496', 'P86479', ...",NaN,"[P86481, P86496, P86479, P86480, P86478]"
19521,ENSP00000334197,386678,29.599539,386678,"{'Swiss-Prot': ['P60412', 'P60411']}",NaN,"[P60412, P60411]"


In [65]:
# Step 1: Filter the rows where uniprot_ids has more than one element
filtered_df = results_df[results_df['uniprot_ids'].apply(lambda x: len(x) > 1 if isinstance(x, list) else False)]

# Step 2: Flatten and extract all IDs into a set
uniprot_id_set = set([item for sublist in filtered_df['uniprot_ids'] for item in sublist])


In [68]:
disease_gene_set = set([item for sublist in dga_combined['uniport'] for item in sublist])

In [ ]:
len(uniprot_id_set),len(disease_gene_set),len(uniprot_id_set&disease_gene_set)
################ we checked the one to more cases, none of issue genes related to disease, so here we just ignore this probelm

(105, 41, 0)

## biocpncept mapping

In [1]:
from gensim.models import KeyedVectors
import os, sys, json, numpy as np

In [2]:
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/bioconcept/concept_fast.json') as json_file:  
    concept_vectors = json.load(json_file)
print('load', len(concept_vectors), 'concepts')
gene_ids = [key for key, value in concept_vectors.items() if key.startswith('Gene')]
gene_unnest = dict()
all_genes = []
for i in gene_ids:
    all_genes.extend(i.split('_')[1:])
    for j in i.split('_')[1:]:
        gene_unnest[j] = i

load 402712 concepts


In [1]:
bio_ids_map = get_map_df(all_genes,'entrezgene')

NameError: name 'get_map_df' is not defined

In [ ]:
bio_ids_map['row_key'] = bio_ids_map["query"].map(gene_unnest)

## gene2vec

In [97]:
path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/pre_processed_features/expression_emb/gene2vec_dim_200_iter_9.txt'

# Read the data from the text file
with open(path, 'r') as file:
    lines = file.readlines()

# Create a dictionary to store the data
data_dict = {}

# Loop through each line and capture the information in the dictionary
for line in lines:
    parts = line.strip().split('\t')
    key = parts[0]
    values = list(map(float, parts[1].split()))
    data_dict[key] = values

# Convert the dictionary to a DataFrame
gene2vec_df = pd.DataFrame.from_dict(data_dict, orient='index')


In [98]:
gene2vec_ids_map = get_map_df(gene2vec_df.index,'symbol')

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
721 input query terms found dup hits:	[('PLAC4', 2), ('COPG2IT1', 2), ('DLEU2L', 2), ('COX6CP2', 2), ('GABARAPL3', 2), ('NBPF25P', 2), ('P
4900 input query terms found no hit:	['C6orf226', 'HIST1H2BN', 'H1FX-AS1', 'LOC284014', 'PAPD7', 'LINC01184', 'LOC101929964', 'FAM49B', '


## uniport T5

In [ ]:
# Path to your .h5 file
file_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/pre_processed_features/seq_emb/per-protein.h5'

# Open and read file safely
with h5py.File(file_path, 'r') as f:
    ids = list(f.keys())  # list() if you want to use it later

## get interesctions of 2019

In [57]:
ppi_set = set()

for values in ppi_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)  # Add the single value


In [59]:
bio_set = set()

for values in bio_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        bio_set.update(values)  # Add all elements in the list
    else:
        bio_set.add(values)  # Add the single value

In [61]:
uniport_set = set(ids)

In [ ]:
inter_2019 = uniport_set & bio_set
inter_2019 = inter_2019 & ppi_set
len(inter_2019)

with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/inter_2019.pkl', 'wb') as f:
    pickle.dump(inter_2019, f)

15753

## intersection 2017

In [66]:
ppi_emb = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2016_emb.csv')
ppi_ids_map = get_map_df(ppi_emb['string_id'].str.split('.').str[1],'ensembl.protein')

ppi_set = set()
for values in ppi_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)  # Add the single value

gene2vec_set = set()
for values in gene2vec_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        gene2vec_set.update(values)  # Add all elements in the list
    else:
        gene2vec_set.add(values)  # Add the single value

inter_2017 = uniport_set & gene2vec_set
inter_2017 = inter_2017 & ppi_set
len(inter_2017)

with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/inter_2017.pkl', 'wb') as f:
    pickle.dump(inter_2017, f)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1803 input query terms found no hit:	['ENSP00000006101', 'ENSP00000035383', 'ENSP00000053469', 'ENSP00000205890', 'ENSP00000207437', 'ENS


In [67]:
len(inter_2017)

15397

## run esm.py to get avaliable esm2 features

In [ ]:


# Load inter_2017
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/inter_2017.pkl', 'rb') as f:
    inter_2017 = pickle.load(f)

# Load inter_2019
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/inter_2019.pkl', 'rb') as f:
    inter_2019 = pickle.load(f)

additional = inter_2017 - (inter_2017 & inter_2019)
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/inter_2017_additional.pkl', 'wb') as f:
    pickle.dump(additional, f)


## map esm2 features

In [11]:
esm2df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/esmfold/protein_embeddings.csv')

In [12]:
esm2df2 = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/esmfold/protein_embeddings_add.csv')


In [13]:
esm_proteins_df = esm2df[~esm2df['string_id'].str.startswith('ENSP')]

In [14]:
esm_proteins_df = pd.concat([esm_proteins_df, esm2df2], ignore_index=True)


In [15]:
esm2df[~esm2df['string_id'].str.startswith('ENSP')].head(3)

,string_id,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_1270,feature_1271,feature_1272,feature_1273,feature_1274,feature_1275,feature_1276,feature_1277,feature_1278,feature_1279
0,Q96P47,-0.013248,-0.004597,0.009793,-0.025982,-0.024738,-0.111435,0.021140,-0.004681,0.049816,...,0.033646,0.003690,-0.065471,0.046642,0.001623,-0.033525,0.044744,-0.075636,0.021780,0.043899
1,A0A0A6YYL3,-0.001943,-0.061321,0.012272,0.010966,-0.004723,-0.100735,0.038466,0.044782,0.000453,...,0.029823,0.048941,-0.120007,-0.023746,0.002480,0.029952,0.028941,-0.026764,-0.012888,-0.040472
2,Q9UL59,-0.057522,-0.038986,0.034499,0.005668,-0.059328,0.026614,0.043679,0.098368,-0.049743,...,0.055876,0.004029,-0.114963,-0.007484,0.010551,0.072261,-0.035139,0.060028,0.078632,0.000347


In [8]:
esm2df2.head(3)

,string_id,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_1270,feature_1271,feature_1272,feature_1273,feature_1274,feature_1275,feature_1276,feature_1277,feature_1278,feature_1279
0,Q8TF21,0.011556,-0.068780,0.063281,-0.013700,0.000587,-0.074444,0.077113,0.010546,0.067425,...,-0.009362,0.054958,-0.179599,-0.054564,0.019109,-0.153406,-0.093485,0.004940,0.093885,0.088577
1,Q8N7B6,-0.020235,-0.000792,0.026569,0.064804,-0.045662,-0.078842,0.117078,-0.081355,0.016124,...,0.039452,0.014842,-0.129878,0.038432,0.010646,-0.017470,0.012853,-0.120761,0.038448,0.088033
2,Q8NGS1,0.016225,-0.029914,-0.057023,0.009274,-0.035819,-0.114321,0.174813,0.173048,0.072618,...,-0.072208,0.084008,-0.207693,-0.030752,-0.061074,0.127716,0.123085,-0.070002,-0.129622,-0.071846


In [16]:
esm_proteins_df

,string_id,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_1270,feature_1271,feature_1272,feature_1273,feature_1274,feature_1275,feature_1276,feature_1277,feature_1278,feature_1279
0,Q96P47,-0.013248,-0.004597,0.009793,-0.025982,-0.024738,-0.111435,0.021140,-0.004681,0.049816,...,0.033646,0.003690,-0.065471,0.046642,0.001623,-0.033525,0.044744,-0.075636,0.021780,0.043899
1,A0A0A6YYL3,-0.001943,-0.061321,0.012272,0.010966,-0.004723,-0.100735,0.038466,0.044782,0.000453,...,0.029823,0.048941,-0.120007,-0.023746,0.002480,0.029952,0.028941,-0.026764,-0.012888,-0.040472
2,Q9UL59,-0.057522,-0.038986,0.034499,0.005668,-0.059328,0.026614,0.043679,0.098368,-0.049743,...,0.055876,0.004029,-0.114963,-0.007484,0.010551,0.072261,-0.035139,0.060028,0.078632,0.000347
3,Q9NUL3,-0.043257,-0.060126,0.025660,0.016698,-0.067359,-0.069697,0.056724,0.040876,0.065235,...,0.107543,-0.092860,-0.130085,0.010876,-0.014858,-0.133465,0.004087,-0.140768,0.101796,0.136234
4,P29017,-0.042537,0.018108,-0.024800,0.187537,-0.102220,-0.168188,0.095551,0.103134,0.037637,...,-0.018744,-0.053489,-0.259134,-0.035090,-0.052050,0.060877,0.138514,-0.096195,-0.009238,0.049916
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16668,Q6PGQ1,-0.022587,-0.043671,-0.009336,0.042837,0.117363,0.019853,0.086841,0.041629,-0.104633,...,0.020182,0.010726,-0.072727,0.107172,0.022719,0.081213,-0.035215,-0.093413,0.020586,0.060609
16669,Q9BQ13,0.057486,0.021183,-0.012304,0.086037,-0.167446,0.015222,0.105242,0.067347,0.013195,...,0.068912,-0.035569,-0.131765,0.047576,0.005940,-0.018026,0.033474,-0.064239,0.034951,0.104081
16670,Q6ZNR0,0.062209,-0.057863,0.034092,0.007490,0.049234,0.009761,0.054670,0.150629,0.099835,...,0.001409,-0.048247,-0.243057,0.121502,-0.004287,-0.037825,-0.032401,-0.019945,0.059332,0.085197
16671,H3BR10,0.017607,-0.068596,0.020347,0.037648,0.001132,-0.016215,-0.005079,0.146299,0.018562,...,0.067755,-0.054596,-0.102110,0.112962,0.035316,-0.041229,-0.052586,0.034249,-0.104639,0.058535


In [2]:
# Load inter_2017
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/inter_2017.pkl', 'rb') as f:
    inter_2017 = pickle.load(f)

# Load inter_2019
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/inter_2019.pkl', 'rb') as f:
    inter_2019 = pickle.load(f)

In [16]:
final_2019 = set(esm_proteins_df['string_id'].tolist())&inter_2019

In [17]:
final_2017 = set(esm_proteins_df['string_id'].tolist())&inter_2017


In [18]:
len(final_2017),len(final_2019)

(15392, 15747)

In [19]:
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/final_2017.pkl', 'wb') as f:
    pickle.dump(final_2017, f)
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/final_2019.pkl', 'wb') as f:
    pickle.dump(final_2019, f)

# clean faetures

In [17]:
esm_proteins_df.rename(columns=lambda x: x.replace('dim_', 'feature_') if x.startswith('dim_') else x, inplace=True)
esm_proteins_df.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/esmfold/uniport_esm2.csv',index=False)

### ppi

In [95]:
ppi_emb = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2016_emb.csv')
ppi_ids_map = get_map_df(ppi_emb['string_id'].str.split('.').str[1],'ensembl.protein')

ppi_set = set()
for values in ppi_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)

string_ids = []
one2more = []
more2one = []  # to collect subdfs with multiple or zero matches

for uniport_ids in list(ppi_set):
    subdf = ppi_ids_map[ppi_ids_map['uniprot_ids'].str.contains(uniport_ids, na=False)]
    
    if len(subdf) == 1:
        if isinstance(subdf['uniprot_ids'], list) and len(values) > 1:
            one2more.append(subdf)
        else:
            string_ids.append(uniport_ids)
    else:
        more2one.append(subdf)

more2one_df = pd.concat(more2one, ignore_index=True)

# Prepare list to store the results
aggregated_rows = []

# Iterate over each UniProt ID group
for protein_id, subdf in more2one_df.groupby('uniprot_ids'):
    # Get list of ENSP IDs
    ensp_ids = subdf['query'].tolist()

    # Add '9606.' prefix to each ENSP ID
    ensp_ids = ['9606.' + ensp_id for ensp_id in ensp_ids]

    # Select corresponding rows from ppi_emb where 'string_id' is in ensp_ids
    matched_ppi = ppi_emb[ppi_emb['string_id'].isin(ensp_ids)]

    if not matched_ppi.empty:
        # Calculate the mean of all feature columns (exclude 'string_id')
        mean_features = matched_ppi.drop(columns=['string_id']).mean()

        # Create a new row with UniProt ID and the averaged features
        mean_features['string_id'] = protein_id

        # Add to the results list
        aggregated_rows.append(mean_features)

# Convert the list of Series into a DataFrame
aggregated_df = pd.DataFrame(aggregated_rows)

# Optional: Reorder columns to have 'uniprot_id' first
cols = ['string_id'] + [col for col in aggregated_df.columns if col != 'string_id']
aggregated_df = aggregated_df[cols]

# Step 1: Prepare ENSP IDs with '9606.' prefix
ensp_ids = ['9606.' + ensp_id for ensp_id in ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)]['query'].tolist()]

# Step 2: Select matching rows from ppi_emb
other_ppi = ppi_emb[ppi_emb['string_id'].isin(ensp_ids)].copy()

# Step 3: Map 'string_id' back to 'uniprot_ids'
# First, create mapping from ENSP ID with '9606.' prefix to UniProt ID
ensp_to_uniprot = ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)].set_index('query')['uniprot_ids'].to_dict()

# Apply mapping to refill 'string_id' with corresponding UniProt ID
other_ppi['string_id'] = other_ppi['string_id'].apply(lambda x: ensp_to_uniprot[x.replace('9606.', '')])

ppi_df = pd.concat([other_ppi, aggregated_df], ignore_index=True)

ppi_df.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/uniport_ppi_2017.csv',index = False)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1803 input query terms found no hit:	['ENSP00000006101', 'ENSP00000035383', 'ENSP00000053469', 'ENSP00000205890', 'ENSP00000207437', 'ENS


### bioconcept

In [ ]:
ppi_set = set()
for values in bio_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)

string_ids = []
one2more = []
more2one = []  # to collect subdfs with multiple or zero matches

for uniport_ids in list(ppi_set):
    subdf = bio_ids_map[bio_ids_map['uniprot_ids'].str.contains(uniport_ids, na=False)]
    
    if len(subdf) == 1:
        if isinstance(subdf['uniprot_ids'], list) and len(values) > 1:
            one2more.append(subdf)
        else:
            string_ids.append(uniport_ids)
    else:
        more2one.append(subdf)

more2one_df = pd.concat(more2one, ignore_index=True)

In [ ]:
gene_ids_features = {key:concept_vectors[key] for key in bio_ids_map['row_key']}
gene_ids_features_df = pd.DataFrame.from_dict(gene_ids_features, orient='index')
gene_ids_features_df.columns = [f'feature_{i}' for i in range(100)]

In [11]:


# Prepare list to store the results
aggregated_rows = []

# Iterate over each UniProt ID group
for protein_id, subdf in more2one_df.groupby('uniprot_ids'):
    # Get list of ENSP IDs
    ensp_ids = subdf['row_key'].tolist()

    # Select corresponding rows from ppi_emb where 'string_id' is in ensp_ids
    matched_ppi = gene_ids_features_df[gene_ids_features_df.index.isin(ensp_ids)]

    if not matched_ppi.empty:
        # Calculate the mean of all feature columns (exclude 'string_id')
        mean_features = matched_ppi.mean()

        # Create a new row with UniProt ID and the averaged features
        mean_features['string_id'] = protein_id

        # Add to the results list
        aggregated_rows.append(mean_features)

# Convert the list of Series into a DataFrame
aggregated_df = pd.DataFrame(aggregated_rows)

# Optional: Reorder columns to have 'uniprot_id' first
cols = ['string_id'] + [col for col in aggregated_df.columns if col != 'string_id']
aggregated_df = aggregated_df[cols]

# Step 1: Prepare ENSP IDs with '9606.' prefix
ensp_ids = [ensp_id for ensp_id in bio_ids_map[bio_ids_map['uniprot_ids'].isin(string_ids)]['row_key'].tolist()]

# Step 2: Select matching rows from ppi_emb
other_ppi = gene_ids_features_df[gene_ids_features_df.index.isin(ensp_ids)].copy()

# Step 3: Map 'string_id' back to 'uniprot_ids'
ensp_to_uniprot = bio_ids_map[bio_ids_map['uniprot_ids'].isin(string_ids)].set_index('row_key')['uniprot_ids'].to_dict()

# Apply mapping to refill 'string_id' with corresponding UniProt ID
other_ppi['string_id'] = other_ppi.index.map(lambda x: ensp_to_uniprot[x])

bio_emb_df = pd.concat([other_ppi, aggregated_df], ignore_index=True)

new_columns = ['string_id'] + [f'feature_{i}' for i, col in enumerate(bio_emb_df.columns) if col != 'string_id']

# Reorder the DataFrame so that 'string_id' is the first column
bio_emb_df = bio_emb_df[['string_id'] + [col for col in bio_emb_df.columns if col != 'string_id']]
bio_emb_df.columns = new_columns


In [13]:
bio_emb_df.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/bioconcept/uniport_bio_emb.csv',index = False)

### gene2vec

In [ ]:
ppi_set = set()
for values in gene2vec_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)

string_ids = []
one2more = []
more2one = []  # to collect subdfs with multiple or zero matches

for uniport_ids in list(ppi_set):
    subdf = gene2vec_ids_map[gene2vec_ids_map['uniprot_ids'].str.contains(uniport_ids, na=False)]
    
    if len(subdf) == 1:
        if isinstance(subdf['uniprot_ids'], list) and len(values) > 1:
            one2more.append(subdf)
        else:
            string_ids.append(uniport_ids)
    else:
        more2one.append(subdf)

more2one_df = pd.concat(more2one, ignore_index=True)

# Prepare list to store the results
aggregated_rows = []

# Iterate over each UniProt ID group
for protein_id, subdf in more2one_df.groupby('uniprot_ids'):
    # Get list of ENSP IDs
    ensp_ids = subdf['query'].tolist()

    # Select corresponding rows from ppi_emb where 'string_id' is in ensp_ids
    matched_ppi = gene2vec_df[gene2vec_df.index.isin(ensp_ids)]

    if not matched_ppi.empty:
        # Calculate the mean of all feature columns (exclude 'string_id')
        mean_features = matched_ppi.mean()

        # Create a new row with UniProt ID and the averaged features
        mean_features['string_id'] = protein_id

        # Add to the results list
        aggregated_rows.append(mean_features)

# Convert the list of Series into a DataFrame
aggregated_df = pd.DataFrame(aggregated_rows)

# Optional: Reorder columns to have 'uniprot_id' first
cols = ['string_id'] + [col for col in aggregated_df.columns if col != 'string_id']
aggregated_df = aggregated_df[cols]

# Step 1: Prepare ENSP IDs with '9606.' prefix
ensp_ids = [ensp_id for ensp_id in gene2vec_ids_map[gene2vec_ids_map['uniprot_ids'].isin(string_ids)]['query'].tolist()]

# Step 2: Select matching rows from ppi_emb
other_ppi = gene2vec_df[gene2vec_df.index.isin(ensp_ids)].copy()

# Step 3: Map 'string_id' back to 'uniprot_ids'
# First, create mapping from ENSP ID with '9606.' prefix to UniProt ID
ensp_to_uniprot = gene2vec_ids_map[gene2vec_ids_map['uniprot_ids'].isin(string_ids)].set_index('query')['uniprot_ids'].to_dict()

# Apply mapping to refill 'string_id' with corresponding UniProt ID
other_ppi['string_id'] = other_ppi.index.map(lambda x: ensp_to_uniprot[x])

gene2vec_emb_df = pd.concat([other_ppi, aggregated_df], ignore_index=True)

new_columns = ['string_id'] + [f'feature_{i}' for i, col in enumerate(gene2vec_emb_df.columns) if col != 'string_id']

# Reorder the DataFrame so that 'string_id' is the first column
gene2vec_emb_df = gene2vec_emb_df[['string_id'] + [col for col in gene2vec_emb_df.columns if col != 'string_id']]
gene2vec_emb_df.columns = new_columns

gene2vec_emb_df.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/pre_processed_features/expression_emb/gene2vec_emb.csv',index = False)

### uniport

In [121]:
all_ids = final_2017 | final_2019

In [ ]:
import h5py

# Replace 'your_file.h5' with the path to your .h5 file
file_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/pre_processed_features/seq_emb/per-protein.h5'
with h5py.File(file_path, 'r') as f:
    # Prepare a list to collect data
    collected_data = []
    index_labels = []

    for id in all_ids:
        if id in f:
            # Retrieve data associated with the current id from the file
            data = f[id][:]
            # Collect data and corresponding index label
            collected_data.append(data)
            index_labels.append(id)

    # If any data was collected, create DataFrame
    if collected_data:
        # Assumes each data is a 1D array, concatenate them into a 2D array
        data_matrix = pd.DataFrame(collected_data, index=index_labels)
data_matrix.columns = [f'feature_{i}' for i in data_matrix.columns]
data_matrix = data_matrix.reset_index().rename(columns={'index': 'string_id'})

data_matrix.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/pre_processed_features/seq_emb/uniport_emb.csv',index = False)

### dga data

In [14]:
dga_combined = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/dga_all_pub.csv')

In [ ]:
import ast
# Container for expanded rows
expanded_rows = []

for idx, row in dga_combined.iterrows():
    # Safely convert the string to a list
    uniport_list = ast.literal_eval(row['uniport'])  # e.g., ['A6NC98,B2RTU8']
    
    # The first element might still contain multiple UniProt IDs separated by commas
    # So we need to split them
    all_uniprots = []
    for entry in uniport_list:
        all_uniprots.extend([u.strip() for u in entry.split(',')])  # Split by comma, strip spaces

    # Duplicate row for each UniProt ID
    for uniprot in all_uniprots:
        new_row = row.copy()
        new_row['uniport'] = uniprot
        expanded_rows.append(new_row)

# Create the expanded dataframe
expanded_df = pd.DataFrame(expanded_rows)



In [18]:
expanded_df['string_id'] = expanded_df['uniport']
expanded_df.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/dga_time_uniport.csv',index = False)

In [19]:
import pandas as pd

In [20]:
expanded_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/dga_time_uniport.csv')

In [24]:
# Safe and clean solution
expanded_df['first_pub_year'] = pd.to_numeric(expanded_df['first_pub_year'], errors='coerce').astype('Int64')



In [26]:
expanded_df.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/dga_time_uniport.csv',index = False)


# check final ids

In [10]:

mg = mygene.MyGeneInfo()
# gene_names = [stringId2name.get(sid) for sid in input_stringids if stringId2name.get(sid) is not None]
result = mg.querymany(list(inter_2019), scopes='uniprot', fields='symbol', species='human')
# Extract gene symbols
gene_names = [entry['symbol'] for entry in result if 'symbol' in entry]

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
114 input query terms found dup hits:	[('O60449', 2), ('P59665', 2), ('Q6FI13', 2), ('P62805', 10), ('P0DN86', 3), ('Q9NS26', 2), ('P50391


In [11]:
for item_list in result:
    if 'symbol' not in item_list.keys():
        print(item_list)


## uniport to symbol

In [2]:
# Load inter_2017
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/inter_2017.pkl', 'rb') as f:
    inter_2017 = pickle.load(f)

# Load inter_2019
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/id_maps/inter_2019.pkl', 'rb') as f:
    inter_2019 = pickle.load(f)

In [3]:
all_uniport = inter_2017 | inter_2019

In [ ]:
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/uniport_id/uni2name.txt', 'w') as f:
    for item in all_uniport:
        f.write(item + '\n')
############ then using uniport id mapping to get mapped ids

In [38]:
uni2name = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/uniport_id/uni2name.txt', sep= '\t')

In [39]:
dupilcated = uni2name[uni2name['From'].duplicated(keep=False)]
len(dupilcated)

157

In [40]:
dupilcated

,From,To
950,Q9H3K6,BOLA2
951,Q9H3K6,BOLA2B
1016,Q8IZN7,DEFB107A
1017,Q8IZN7,DEFB107B
1664,Q9ULZ0,TP53TG3
...,...,...
16464,Q6FI13,H2AC19
16599,Q9Y6F8,CDY1
16600,Q9Y6F8,CDY1B
16608,Q96JG8,MAGED4


In [32]:
uni2name

,From,To
0,O94933,SLITRK3
1,Q3SY69,ALDH1L2
2,Q9NSB8,HOMER2
3,Q9NY43,BARHL2
4,Q5JPI9,EEF1AKMT2
...,...,...
16781,P03952,KLKB1
16782,P35612,ADD2
16783,Q9NWN3,FBXO34
16784,Q9BUU2,METTL22


In [33]:
uni2name_dict = dict()
for uni, subdf in uni2name.groupby('From'):
    uni2name_dict[uni] = subdf['To'].tolist()

In [34]:
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/uniport_id/uni2name.pkl', 'wb') as f:
    pickle.dump(uni2name_dict, f)

In [47]:
with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/uniport_id/uni2name.pkl', 'rb') as file:
    uni2name_dict = pickle.load(file)

In [48]:
input_ids = ['O94933', 'Q9ULZ0']
gene_names = set()

for unid in input_ids:
    gene_list = uni2name_dict.get(unid, [])  # use .get() to avoid KeyError
    gene_names.update(gene_list)  # update adds all elements from the list


In [50]:
list(gene_names)

['TP53TG3',
 'TP53TG3B',
 'TP53TG3F',
 'TP53TG3C',
 'TP53TG3D',
 'TP53TG3E',
 'SLITRK3']